In [ ]:
# PDF Q&A Assistant — Retrieval-Augmented Generation (RAG)

## Project Overview

This notebook builds a **PDF Q&A assistant** that can answer questions about any PDF document with **cited context**. Instead of pasting an entire document into an LLM prompt (which breaks for large documents), we use a technique called **Retrieval-Augmented Generation (RAG)** — the dominant pattern behind tools like ChatGPT with file uploads, Perplexity, and enterprise knowledge assistants.

**What we build:**  
A pipeline that loads PDFs → chunks them → embeds chunks into vectors → stores them in a vector database → retrieves the most relevant chunks at query time → feeds them to an LLM to produce grounded, cited answers.

**Stack:**
| Component | Tool | Why |
|-----------|------|-----|
| PDF parsing | PyPDF | Reliable, page-aware text extraction |
| Orchestration | LangChain | Industry-standard RAG framework |
| LLM | Ollama (local) | No API keys, runs offline, free |
| Embeddings | Ollama nomic-embed-text | Strong open-source embeddings |
| Vector store | ChromaDB | Lightweight, in-process, no setup |

**Prerequisite:** [Ollama](https://ollama.com/) must be installed and running locally.

## 2 — Learning Goals

By the end of this notebook you will be able to:

1. **Explain RAG** — what it is, why it exists, and when to use it vs. just prompting an LLM directly
2. **Build a document ingestion pipeline** — load PDFs, chunk text with overlap, embed chunks, and store vectors
3. **Implement semantic retrieval** — use a vector database to find the most relevant chunks for a user query
4. **Chain retrieval + generation** — connect a retriever to an LLM with a citation-aware prompt template
5. **Compare naive prompting vs. RAG** — understand the tradeoffs in quality, cost, and reliability
6. **Tune the pipeline** — adjust chunk size, overlap, number of retrieved chunks, and embedding model
7. **Identify common failure modes** — hallucination, poor chunking, embedding mismatch, and how to debug them

## 3 — Problem Statement

**Problem:** You have a 15-page research paper (or legal contract, manual, report). You want to ask specific questions and get accurate answers with page citations — without reading the whole thing.

**Why can't we just paste it into ChatGPT?**
- Documents can exceed the LLM's context window (or be expensive to process fully)
- Even if it fits, the LLM may ignore information buried in the middle ("lost in the middle" effect)
- No citation mechanism — you don't know which part of the document the answer came from
- Every query re-sends the entire document (wasteful for repeated Q&A)

**RAG solves this** by pre-processing the document once, then retrieving only the relevant 3-5 chunks per question.

In [ ]:
## 4 — Why This Project Matters

RAG is the **most widely deployed LLM pattern in production** — far more common than fine-tuning.

**Real-world applications:**
- **Enterprise search** — employees querying internal policy docs, HR manuals, engineering specs
- **Legal tech** — lawyers querying case law and contracts
- **Healthcare** — clinicians querying drug interaction databases and clinical guidelines
- **Customer support** — bots answering questions from product documentation
- **Research** — scientists querying paper collections

**Why learn RAG before fine-tuning?**
- RAG is cheaper (no training compute)
- RAG is faster to deploy (minutes vs. hours/days)
- RAG is more maintainable (update docs, not model weights)
- RAG produces more trustworthy answers (grounded in retrieved text)
- ~80% of enterprise LLM use cases are solved by RAG, not fine-tuning

## Step 2 — Initialize the LLM and Embedding Model

We use **Ollama** to run models locally — no API keys needed.

- **LLM**: `qwen3.5:9b` — a strong general-purpose model
- **Embeddings**: `nomic-embed-text-v2-moe` — converts text to numerical vectors

> **What are embeddings?** They convert text into a list of numbers (a vector) such that
> similar meanings produce similar vectors. This lets us do "semantic search" — finding
> text by *meaning* rather than exact word matches.

In [ ]:
# Initialize the LLM — runs locally via Ollama
llm = ChatOllama(
    model="qwen3.5:9b",
    temperature=0.3,  # Lower = more focused/factual answers
)

# Initialize the embedding model
embeddings = OllamaEmbeddings(
    model="nomic-embed-text-v2-moe",
)

# Quick test
test_vector = embeddings.embed_query("Hello world")
print(f"Embedding dimension: {len(test_vector)}")
print(f"First 5 values: {test_vector[:5]}")

## Step 3 — Load a PDF Document

We'll download a sample PDF (a public research paper) to demonstrate.
`PyPDFLoader` extracts text from each page and keeps page numbers as metadata.

In [ ]:
import urllib.request

# Download a sample PDF — the "Attention Is All You Need" paper
pdf_url = "https://arxiv.org/pdf/1706.03762v7"
pdf_path = Path("sample_paper.pdf")

if not pdf_path.exists():
    print("Downloading sample PDF...")
    urllib.request.urlretrieve(pdf_url, pdf_path)
    print(f"Downloaded: {pdf_path} ({pdf_path.stat().st_size / 1024:.0f} KB)")
else:
    print(f"PDF already exists: {pdf_path}")

# Load the PDF — each page becomes a separate Document
loader = PyPDFLoader(str(pdf_path))
pages = loader.load()

print(f"\nLoaded {len(pages)} pages")
print(f"\n--- Page 1 preview (first 500 chars) ---")
print(pages[0].page_content[:500])
print(f"\nMetadata: {pages[0].metadata}")

## Step 4 — Chunk the Document

LLMs have limited context windows, and embedding models work best on smaller pieces.
We split the text into overlapping chunks.

### Why overlap?
Without overlap, a sentence split at a chunk boundary loses context.
Overlap ensures the boundary text appears in at least two chunks.

```
Chunk 1: [==========]
Chunk 2:        [==========]     ← overlap region
Chunk 3:               [==========]
```

In [ ]:
# Create a text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,      # Each chunk ≈ 1000 characters
    chunk_overlap=200,    # 200 chars overlap between consecutive chunks
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""],  # Try to split at natural boundaries
)

# Split all pages into chunks
chunks = text_splitter.split_documents(pages)

print(f"Split {len(pages)} pages into {len(chunks)} chunks")
print(f"\n--- Chunk 0 ---")
print(f"Content: {chunks[0].page_content[:300]}...")
print(f"Metadata: {chunks[0].metadata}")
print(f"\n--- Chunk 5 ---")
print(f"Content: {chunks[5].page_content[:300]}...")
print(f"Metadata: {chunks[5].metadata}")

## Step 5 — Create the Vector Store

Now we embed every chunk and store the vectors in **ChromaDB**.
ChromaDB is a lightweight vector database that runs in-process (no server needed).

```
"The Transformer architecture..." → [0.12, -0.34, 0.56, ...] → stored in ChromaDB
"Attention mechanism allows..."   → [0.15, -0.31, 0.58, ...] → stored in ChromaDB
```

In [ ]:
# Create vector store from chunks
# This embeds all chunks and stores them — takes a minute on first run
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db",  # Persists to disk so we don't re-embed
    collection_name="pdf_qa",
)

print(f"Vector store created with {vectorstore._collection.count()} vectors")

## Step 6 — Test Retrieval

Before building the full Q&A chain, let's verify that retrieval works.
We search for chunks most similar to a query.

In [ ]:
# Create a retriever — fetches top 4 most relevant chunks
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},  # Return top 4 chunks
)

# Test retrieval
query = "What is the Transformer architecture?"
retrieved_docs = retriever.invoke(query)

print(f"Query: '{query}'")
print(f"Retrieved {len(retrieved_docs)} chunks:\n")

for i, doc in enumerate(retrieved_docs):
    page = doc.metadata.get('page', '?')
    print(f"--- Chunk {i+1} (Page {page}) ---")
    print(doc.page_content[:200])
    print()

## Step 7 — Build the Q&A Chain with Citations

Now we connect the retriever to the LLM with a custom prompt that asks
the model to cite which page its answer comes from.

### The RAG Prompt Pattern
```
System: You are a helpful assistant. Answer based ONLY on the provided context.
        If you don't know, say so. Cite page numbers.

Context: {retrieved chunks}

Question: {user question}
```

In [ ]:
# Custom prompt template with citation instructions
prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template="""You are a helpful research assistant. Answer the question based ONLY on
the following context extracted from a PDF document. If the context doesn't contain
enough information to answer, say "I don't have enough information to answer this."

Always cite the page number(s) where you found the information using [Page X] format.

Context:
{context}

Question: {question}

Answer (with citations):"""
)

# Build the RetrievalQA chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",       # "stuff" = put all retrieved docs into the prompt
    retriever=retriever,
    return_source_documents=True,  # Also return the source chunks
    chain_type_kwargs={"prompt": prompt_template},
)

print("Q&A chain ready!")

## Step 8 — Ask Questions!

Let's put it all together and ask questions about the PDF.

In [ ]:
def ask(question: str) -> None:
    """Ask a question about the PDF and display the answer with sources."""
    print(f"❓ Question: {question}")
    print("-" * 60)
    
    result = qa_chain.invoke({"query": question})
    
    print(f"\n💡 Answer:\n{result['result']}")
    
    print(f"\n📄 Sources used:")
    seen_pages = set()
    for doc in result["source_documents"]:
        page = doc.metadata.get("page", "?")
        if page not in seen_pages:
            seen_pages.add(page)
            print(f"  - Page {page}: \"{doc.page_content[:80]}...\"")
    print("=" * 60)

# Try several questions
ask("What is the main contribution of this paper?")

In [ ]:
ask("How does multi-head attention work?")

In [ ]:
ask("What were the BLEU scores achieved on the English-to-German translation task?")

In [ ]:
# Try a question that CAN'T be answered from this paper
ask("What is the capital of France?")

## 🧠 Key Concepts Recap

| Concept | What it does |
|---------|-------------|
| **Document Loader** | Extracts text from PDFs (one Document per page) |
| **Text Splitter** | Breaks text into manageable, overlapping chunks |
| **Embeddings** | Converts text chunks into numerical vectors |
| **Vector Store** | Stores and indexes vectors for fast similarity search |
| **Retriever** | Finds the most relevant chunks for a query |
| **RetrievalQA** | Chains retriever + LLM to answer questions |
| **Prompt Template** | Structures the instruction sent to the LLM |

## 🔧 Tuning Tips

- **`chunk_size`**: Larger = more context per chunk but fewer matches. Smaller = more precise but may lack context.
- **`chunk_overlap`**: Increase if you see answers cut off mid-sentence.
- **`k` (retrieval count)**: More chunks = more context but slower and potentially noisier.
- **`temperature`**: Lower (0.1) for factual Q&A; higher (0.7) for creative tasks.